# Practice 43 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — titanic: string ops on real columns

Load `sns.load_dataset('titanic')`.

The `who` column has values `'man'`, `'woman'`, `'child'`. The `embark_town` column has full town names.

1. Use `.str.extract()` to pull the first 3 characters of `embark_town` into a new column `town_code` (e.g. `'Cherbourg'` → `'Che'`).
2. Use `.str.contains()` to find passengers from a town that starts with `'S'`. How many? Use `na=False`.
3. The `deck` column has values like `'A'`, `'B'` etc. Use `.str.extract()` to confirm you can pull the single letter. Then use `np.select()` to add `deck_zone`: `'upper'` (A/B/C), `'middle'` (D/E), `'lower'` (F/G), default `'unknown'` — use `.str.contains()` inside your conditions.
4. Convert `deck_zone` and `town_code` to `category`. What is the survival rate per `deck_zone`? Lean — one groupby, no hints.
5. Among passengers whose `embark_town` contains the letter `'o'` (case-insensitive), what fraction survived?

In [21]:
# Your code here

titanic = sns.load_dataset('titanic')

titanic['town_code'] = titanic['embark_town'].str.extract(r'^(\w{3})')
titanic['embark_town'].str.contains('S', na=False).sum()

titanic['deck'].str.extract(r'(\w{1})')
conds = [titanic['deck'].str.contains('A|B|C', na=False), titanic['deck'].str.contains('D|E', na=False), titanic['deck'].str.contains('F|G', na=False)]
chos = ['upper','middle','lower']
titanic['deck_zone'] = np.select(conds, chos, default= 'unknown')

titanic[['deck_zone','town_code']] = titanic[['deck_zone','town_code']].astype('category')
titanic.groupby('deck_zone', observed=True)['survived'].mean()

titanic.loc[titanic['embark_town'].str.contains('o', case=False,na=False), 'survived'].mean()

np.float64(0.38245219347581555)

---
## Level 2 — spot the bug

Load `sns.load_dataset('mpg')`. Each snippet has one bug — fix it and explain in a comment.

```python
# A — extract the first word of each car name
mpg['make'] = mpg['name'].str.extract(r'\w+')

# B — flag cars whose name contains 'ford' or 'chevy'
mpg['is_american'] = mpg['name'].str.contains('ford') | 'chevy'

# C — extract decade from year
mpg['decade'] = mpg['year'].str.extract(r'(\d{3})')[0] + '0s'

# D — flag cars heavier than average
mpg['heavy'] = mpg['weight'].apply(lambda x: x > mpg['weight'].mean())

# E — get number of unique makes after extracting
mpg['make'] = mpg['name'].str.extract(r'^(\w+)')[0]
mpg['make'].unique().shape
```

In [59]:
mpg = sns.load_dataset('mpg').dropna()

# Your fixes here
mpg['make'] = mpg['name'].str.extract(r'(\w+)')
mpg['is_american'] = mpg['name'].str.contains('ford|chevy')
mpg['decade'] =mpg['model_year'].astype(str).str.extract(r'(\d{1})')[0]+'0s' ## should be this but this df is different 
mpg['heavy'] = mpg['weight'].apply(lambda x: x > mpg['weight'].mean()) ## this is correct, below is the vecterazation 
mpg['heavy'] = mpg['weight']>mpg['weight'].mean()
mpg['make'] = mpg['name'].str.extract(r'^(\w+)')[0]
len(mpg['make'].unique())

36

In [60]:
mpg['decade']

0      70s
1      70s
2      70s
3      70s
4      70s
      ... 
393    80s
394    80s
395    80s
396    80s
397    80s
Name: decade, Length: 392, dtype: object

---
## Level 3 — pipeline on review data

You have a product review dataset. Write a `.pipe()` chain — no `.apply()`.

- **`parse(df)`** — use `.str.extract()` to pull `rating_num` from `rating` (e.g. `'4/5'` → `4`) as integer. Use `.str.contains()` to add `is_verified`: True where `tags` contains `'verified'`. Use `.str.extract()` to pull the `product_code` — the letters before the dash in `product_id`.
- **`classify(df)`** — add `sentiment` with `np.select()`: `'positive'` (rating_num >= 4), `'neutral'` (rating_num == 3), `'negative'` otherwise. Add `word_count` = length of `review_text` split by spaces — use `.str.split().str.len()`. Add `is_detailed`: True where `word_count` > 10.

After the chain:
- What fraction of verified reviews are positive? Use `.loc` and `np.mean()`.
- Which `product_code` has the highest mean `rating_num`? One groupby.
- Use a dict comprehension to build `{sentiment: mean_word_count}` from the groupby result.
- Use a set comprehension to find all `product_code` values that have at least one negative review.

In [58]:
reviews = pd.DataFrame({
    'product_id':   ['WG-001', 'GD-002', 'WG-001', 'DH-003', 'GD-002',
                     'WG-001', 'DH-003', 'GD-002', 'WG-001', 'DH-003'],
    'rating':       ['5/5', '3/5', '4/5', '1/5', '5/5',
                     '2/5', '4/5', '3/5', '5/5', '2/5'],
    'tags':         ['verified,helpful', 'verified', 'helpful', 'verified', 'verified,helpful',
                     'helpful', 'verified', 'helpful', 'verified,helpful', 'verified'],
    'review_text':  [
        'Great product, works perfectly and very fast delivery',
        'It is okay nothing special',
        'Good value for the price I paid',
        'Completely broke after one day total waste of money',
        'Absolutely love it best purchase I have made this year',
        'Not what I expected',
        'Does the job fine for the price',
        'Average product not bad not great',
        'Excellent quality highly recommend to everyone',
        'Poor quality broke quickly very disappointed with this product',
    ]
})

# Your code here
def parse(df):
    df['rating_num'] = df['rating'].str.extract(r'(\d{1})(?:\/)').astype(int)
    df['is_verified'] = df['tags'].str.contains('verified')
    df['product_code'] = df['product_id'].str.extract(r'([a-zA-Z]+)(?:-)')
    return df 

def classify(df):
    conds = [df['rating_num']>=4, df['rating_num']==3]
    chos = ['positive','neutral']
    df['sentiment'] = np.select(conds, chos, default = 'negative')
    df['word_count'] = df['review_text'].str.split(" ").str.len()
    df['is_detailed'] = df['word_count']>10
    return df 


res = reviews.copy().pipe(parse).pipe(classify)

np.mean(res.loc[res['is_verified'],'sentiment']=='positive')

res.groupby('product_code')['rating_num'].mean().idxmax()
g = res.groupby('sentiment')['word_count'].mean()
{s:m for s,m in zip(g.index, g)}

neg_res = res.loc[res['sentiment']=='negative']

{pc for pc in neg_res['product_code']}


{'DH', 'WG'}

You have a product review dataset. Write a `.pipe()` chain — no `.apply()`.

- **`parse(df)`** — use `.str.extract()` to pull `rating_num` from `rating` (e.g. `'4/5'` → `4`) as integer. Use `.str.contains()` to add `is_verified`: True where `tags` contains `'verified'`. Use `.str.extract()` to pull the `product_code` — the letters before the dash in `product_id`.
- **`classify(df)`** — add `sentiment` with `np.select()`: `'positive'` (rating_num >= 4), `'neutral'` (rating_num == 3), `'negative'` otherwise. Add `word_count` = length of `review_text` split by spaces — use `.str.split().str.len()`. Add `is_detailed`: True where `word_count` > 10.

After the chain:
- What fraction of verified reviews are positive? Use `.loc` and `np.mean()`.
- Which `product_code` has the highest mean `rating_num`? One groupby.
- Use a dict comprehension to build `{sentiment: mean_word_count}` from the groupby result.
- Use a set comprehension to find all `product_code` values that have at least one negative review.